# Word Embeddings Introduction

Word embeddings are a type of word representation that allows words to be expressed as vectors in a continuous vector space. These vectors capture semantic menaings and relationshps between words bases on their usage in large text of corpora

## Import and Package installation

Let's install some important packages and libraries

In [ ]:
!pip install --upgrade gensim
!pip install scikit-learn
!pip install matplotlib
!pip install datasets
!pip install plotly

In [ ]:
!pip install fasttext

Let's import some useful libraries

In [ ]:
from google.colab import drive
import os
from gensim.models.word2vec import Word2Vec
import gensim.downloader as api
from gensim.models import FastText, Word2Vec, KeyedVectors
import string
import numpy as np
import pandas as pd
from pandas.core.common import flatten
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from datasets import load_dataset
import re
import csv
import random
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import fasttext
import seaborn as sns
from scipy.spatial.distance import pdist, squareform

## Setup

In [ ]:
drive.mount('/content/drive')
os.chdir(f'/content/drive/MyDrive/Colab Notebooks/NLP/Assignment/datasets')
os.getcwd()

Set to true **HUGGINGFACE** to download the dataset directly from HuggingFace

In [ ]:
HUGGINGFACE = False

In [ ]:
nltk.download('wordnet')

In [ ]:
nltk.download('punkt')

## Utils

Define a dictionary of contractions and their expansions

In [ ]:
contractions_dict = {
    "can't": "cannot",
    "won't": "will not",
    "I'm": "I am",
    "he's": "he is",
    "she's": "she is",
    "it's": "it is",
    "they're": "they are",
    "we're": "we are",
    "you're": "you are",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "I'll": "I will",
    "you'll": "you will",
    "he'll": "he will",
    "she'll": "she will",
    "we'll": "we will",
    "they'll": "they will",
    "I'd": "I would",
    "you'd": "you would",
    "he'd": "he would",
    "she'd": "she would",
    "we'd": "we would",
    "they'd": "they would",
    "there's": "there is",
    "here's": "here is",
    "how's": "how is",
    "let's": "let us",
    "that's": "that is",
    "what's": "what is",
    "who's": "who is",
    "where's": "where is",
    "why's": "why is",

}

The **expand_contractions** function is designed to expand contractions in a given text based on a provided dictionary of contractions and their corresponding expanded forms.

In [ ]:
def expand_contractions(text, contractions_dict=contractions_dict):
    contractions_pattern = re.compile('({})'.format('|'.join(contractions_dict.keys())),
                                      flags=re.IGNORECASE | re.DOTALL)

    def replace(match):
        match_text = match.group(0)
        expanded_text = contractions_dict.get(match_text.lower())
        return expanded_text

    expanded_text = contractions_pattern.sub(replace, text)
    return expanded_text

The function **preprocess_document** is designed to process a given document, performing various optional text preprocessing steps based on the specified parameters. These parameters include:

* expand: If set to True, the function expands contractions in the document (e.g., "can't" to "cannot").
* isSorted: If set to True, the function sorts the processed document alphabetically.
* isSet: If set to True, the function converts the processed document into a set, removing any duplicate elements.
* noStopWords: If set to True, the function removes common English stopwords from the processed document.
* lemmatization: If set to True, the function lemmatizes the words in the processed document using WordNet.

The function begins by expand contractions, removing punctuation from the document using a regular expression. Then, depending on the parameters provided, it applies additional processing steps accordingly. Finally, it returns the processed document.

In [ ]:
def preprocess_document(document, expand=False, noPunctuation=False, isSorted=False, isSet=False, noStopWords=False, lemmatization=False):
    newDocument = document
    if expand:
        newDocument = [expand_contractions(word) for word in newDocument]
    if noPunctuation:
        regex = '[' + string.punctuation + ']'
        newDocument = [re.sub(regex, '', item) for item in newDocument]
    if isSet:
        newDocument = set(newDocument)
    if noStopWords:
        newDocument = [w for w in newDocument if w not in stopwords.words('english')]
    if isSorted:
        newDocument = sorted(newDocument)
    if lemmatization:
        lemmatizer = WordNetLemmatizer()
        newDocument = [lemmatizer.lemmatize(w) for w in newDocument]
    return ' '.join(newDocument)

The upcoming snippet of code demonstrates how to use vector arithmetci with a pre-trained word embedding model to find terms semantically similar to a given concept given the vocabulary of the embedding.
This technique leverages the semantic relationships captured by the word embeddings to explore and understand the conceptual adjustments within the embedding space

In [ ]:
def adjust_term_vector(model,term, term_to_adjust_from, term_to_adjust_to, topN=10):
  term_vec = model.get_vector(term)
  adjust_from_vec = model.get_vector(term_to_adjust_from)
  adjust_to_vec = model.get_vector(term_to_adjust_to)
  adjusted_vec = term_vec + (adjust_from_vec - adjust_to_vec)
  closest_terms = model.most_similar(adjusted_vec, topn=topN)
  return closest_terms

In [ ]:
def plot_PCA_decomposition(model, words):
  words_vec = np.array([model[word] for word in words])
  pca = PCA(n_components=2)
  words_vec_2d = pca.fit_transform(words_vec)
  plt.figure(figsize=(7, 7))
  plt.scatter(words_vec_2d[:, 0], words_vec_2d[:, 1], edgecolors='k', c='g')
  for word, (x, y) in zip(words, words_vec_2d):
      plt.text(x, y, word, fontsize=12)
  plt.show()

# Dataset loading

**Basic Description of the Dataset and its Design**

The dataset in our study was derived from the Anki Medical Curriculum flashcards, a comprehensice collection curated and continually updated by medical students. Thes flashcards cover the full breadth of the medcial curriculum, encompassing subjects suche as anatomy, physiology, pathology and pharmacology. Each flashcard typically includes summaries to facilitate the learning and retention of key medical concepts. To create the dataset, the content was extracted from these flashcards, excluding those containing images.
Then, OpenAI's GPT 3.5- turbo was employed to transfrom the flashcard content into coherent, contextually relevant question-answer pairs.

The dataset consists of over 30,000 questions, each accompanied by a corresponding segment of text serving as its answer. Notably, these answers lack any additional context.



**Dataset Loading**

We initiate dataset download with this code snippet, offering flexibility in data retrieval. Depending on the value of the HUGGINGFACE flag, the script fetches the dataset from Hugging Face's repository or directly loads it from a CSV file stored on Google Drive. This versatile approach simplifies the process of acquiring the necessary data for analysis or modeling tasks.

## Huggining face loading

In [ ]:
if HUGGINGFACE == True:
  raw_datasets =  load_dataset("medalpaca/medical_meadow_medical_flashcards", download_mode="force_redownload")
  raw_dataframe = raw_datasets['train'].to_pandas()
  raw_dataframe.to_csv('MedFlashCards.csv', index=False)


## Drive loading

In [ ]:
if HUGGINGFACE == False:
  medFlashCards_dataframe = pd.read_csv('MedFlashCards.csv')

question_df = pd.read_csv('questions_dataframe.csv')
answer_df = pd.read_csv('answers_dataframe.csv')


# Learning word embeddings

## Data Preprocessing

Data preprocessing is crucial before training word embeddings because it ensures the text data is clean, consistent, and meaningful. Here's why each operation is important:


*   **Remove Punctuation and Special Characters**: Punctuation and special characters do not carry meaningful semantic information for word embeddings and can introduce noise. Removing them helps focus on the actual words and their relationships

*   **Expand Contractions**: Contractions like "can't" or "won't" are expanded to "cannot" and "will not" to standardize the text. This ensures that the model treats them as the same words rather than different tokens, improving the quality of embeddings

*  **Lowercase Words**: Converting all words to lowercase standardizes the text, so "Apple" and "apple" are treated as the same word. This reduces the vocabulary size and improves the embeddings' consistency.

*  **Lemmatization**: Lemmatization reduces words to their base or root form, such as converting "running" to "run." This helps in grouping similar words together, ensuring that the embeddings capture the core meaning rather than different forms of the same word.

By performing these preprocessing steps, the text data becomes cleaner and more uniform, allowing the word embeddings to capture more accurate and meaningful representations of the words.

In [ ]:
train_questions = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(), noPunctuation=True, expand=True, lemmatization=True ))
train_answers = answer_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(), noPunctuation=True, expand=True, lemmatization=True ))

First, we will train word embeddings using the Word2Vec implementation from the Gensim package

In [ ]:
train_questions = list(flatten(train_questions))
train_answers = list(flatten(train_answers))

## Tokenization


In the upcoming code block, the tokenization process is performed. Tokenization involves splitting text into smaller units called tokens, which can be words, subwords, or characters. For word embeddings, this typically means breaking down a text into individual words or subwords that can then be converted into vectors.

In [ ]:
train_questions_tokenized = [word_tokenize(question) for question in train_questions]
train_answers_tokenized = [word_tokenize(answer) for answer in train_answers]
tokenized_sentences = train_questions_tokenized + train_answers_tokenized

## Word2Vec

Word2Vec is a method for generating word embeddings by training a shallow neural network on a large corpus of text data. It learns dense vector representations of words based on their context, capturing semantic relationships between words. These embeddings are widely used in NLP tasks to measure semantic similarity and improve performance in various applications



Finally we have the data in the right format for training Word2Vec, so we can provide it to the algorithm. For parameters, we set:
- the embedding size to be 30,
- the minimum count for any vocabulary term to be 5
- the size of the context window to 10.

In [ ]:
model = Word2Vec(tokenized_sentences, vector_size=30, min_count=5, window=10)

Let's see how big the vocabulary is that Word2Vec ended up using, i.e. how many word vectors did it learn:

In [ ]:
num_vectors = len(model.wv)
print('Number of learned vectors: ', num_vectors)

### Inspecting embeddings and finding similar words

Let's print out one of the vectors to see what looks like now that we have a word2vec model trained:

In [ ]:
term = 'pregnancy'
model.wv[term]

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space.


In [ ]:
term = 'heart'
model.wv.most_similar(term)

In [ ]:
term = 'virus'
model.wv.most_similar(term)

In [ ]:
term = 'antidepressant'
model.wv.most_similar(term)

In [ ]:
term = 'insulin'
model.wv.most_similar(term)


The following snippet of code shows how a word's meaning shifts based on an analogy. Imagine insulin is the base word. We want its meaning to move away from diabetes and closer to pneumonia. The code uses a fancy method (PCA) to create a visual map where closer words have similar meaning

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'
closest_terms = adjust_term_vector(model.wv,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model.wv, words)

### Visualising the embedding vectors using t-SNE

We'll now visualise some of the word vectors in a 3 dimensional space using t-SNE.
The vocabulary of word vectors is quite large (around 25,000). Giving them all to t-SNE will cause it to take far too long to converge. So let's first choose a random subset of 500 terms to show.

In [ ]:
sample = random.sample(list(model.wv.key_to_index), 500)
print(sample)

Now we'll get the word vectors for the sampled terms:

In [ ]:
word_vectors = model.wv[sample]
word_vectors

And we'll provide the vectors to TSNE to fit a model and transform the data to 3 dimensions:

In [ ]:
tsne = TSNE(n_components=3, n_iter=2000)
tsne_embedding = tsne.fit_transform(word_vectors)

Now transform the data into 3 columns (for x, y, and z):

In [ ]:
x, y, z = np.transpose(tsne_embedding)

And generate the 3d plot:

In [ ]:
fig = px.scatter_3d(x=x, y=y, z=z)
fig.update_traces(marker=dict(size=3,line=dict(width=2)))
fig.show()

Well that's a not a particularly interesting 3d plot!
- How about we label some of the points on the graph to see what words they correspond to:

In [ ]:
fig = px.scatter_3d(x=x[:200],y=y[:200],z=z[:200],text=sample[:200])
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

Let's extend the random set of terms with a set of colours to see if they cluster:

In [ ]:
# Add some specific terms to sample:
colours = ['red','green','blue','orange','yellow','purple','pink','cream','brown','black','white','gray']

word_vectors = model.wv[colours+sample]

tsne = TSNE(n_components=3)
tsne_embedding = tsne.fit_transform(word_vectors)

x, y, z = np.transpose(tsne_embedding)

In [ ]:
import plotly.express as px

r = (-200,200)
fig = px.scatter_3d(x=x, y=y, z=z, range_x=r, range_y=r, range_z=r, text=colours + [None] * 500)
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

Note: t-SNE is a stochastic algorithm, so run it a couple of times to see how the visualisation changes.

Have a play around with the visualisation to see whether other sets of terms cluster together.

# Loading Pre-trained Embedding

## Glove-wiki-Gigaword-200

Using GloVe embedddings pre-trained on the Wikipedia 2014 + Gigaword 5 dataset (glove-wiki-gigaword-200) can enhance the performance of question answering systems on our dataset. These embeddings are trained on an extensive corpus, capturing deep semantic and synactic nuances. This pretraining provides a wealth of linguistic knowledge. With 200 dimensions, GloVe embeddings achieve a balance between capturing intricate word relationships and maintaning computational efficiency.
Additionally, employing pretrained embeddings accelerates the training process, as the model starts with rich word representations instead of learning them from scratch. Consequently, glove-wiki-gigaword-200 is an effective, efficient, and practical choice for enhancing question-answering systems across various datasets.

**GloVe-Wiki-Gigaword 200 overview**:


*   **Algorithm**: GloVe which captures global word-word co-occurence statistics
*   **Training Data**: Wikipedia and Gigaword corpus
*   **Vector dimension**: 200-dimensional embeddings




In [ ]:
model_wiki = api.load('glove-wiki-gigaword-200')

Let's see how big the vocabulary:

In [ ]:
print('Vocabulary size of Wikipedia model: ', len(model_wiki))

### Inspecting embeddings and finding similar words

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space

In [ ]:
term = 'virus'
model_wiki.most_similar(term)

This code demonstrates how a word's meaning shifts based on analogies using vector arithmetic, such as "king" to "queen" and "man" to "woman." While a pretrained model on a large open-domain corpus can effectively capture complex word relationships, it often struggles with domain-specific analogies, such as "insulin" to "antibiotics" and "diabetes" to "pneumonia." In the following sections, we will explore how using a domain-specific pretrained model can enhance our performance in these specialized contexts.

In [ ]:
term_vec = 'king'
adjust_from_vec = 'woman'
adjust_to_vec = 'man'
closest_terms = adjust_term_vector(model_wiki,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model_wiki, words)

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'
closest_terms = adjust_term_vector(model_wiki,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model_wiki, words)

## Word2Ve - PMC - 100

Word embeddings create vector representations of text data, but not all embeddings effectively capture clinical, medical, and biomedical information. These embeddings provide a low-dimensional representation of semantic meaning, facilitating efficient text data analysis. While they are useful across various domains, their application to clinical, medical, and biomedical encounter notes presents several challenges. Commonly available pre-trained embeddings are often derived from broad, non-medical text sources, which may not accurately capture the specific word senses, linguistic relationships, or vocabulary needed in a clinical context. To address this, a training corpus based on clinical case reports from the PubMed Open Access subset is used. Consequently, a medical and biomedical pre-trained embedding model was employed.

**Word2Vec-PMC-Open-Access-All-Manuscripts overview**:


*   **Algorithm**: Word2Vec
*   **Training Data**: PMC Open Access All Manuscripts
*   **Vector dimension**: 100-dimensional embeddings

First download and extract the files from each archive

In [ ]:
!tar -xvf w2v_100d_oa_all.tar.gz

In [ ]:
# Load the model
model = Word2Vec.load('W2V_100/w2v_oa_all_100d.bin')

Let's see how big the vocabulary:

In [ ]:
print(f'Size of the vocabulary: {len(model.wv)}')

### Inspecting embeddings and finding similar words

Return 100-dimensional vector representations of each word

In [ ]:
term = 'diabetes'
model.wv.get_vector(term)

In [ ]:
term = 'heart'
model.wv.get_vector(term)

In [ ]:
term = 'pregnancy'
model.wv.get_vector(term)

In [ ]:
model.wv.similarity('myocardial_infarction', 'heart_attack')

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space.


In [ ]:
term = 'delirium'
model.wv.most_similar(term)

In [ ]:
term = 'heart'
model.wv.most_similar(term)

The upcoming snippet of code demonstrates how to use vector arithmetic for word's meaning shifts based on analogies with a pre-trained word embedding model - Word2Vec on PMC Open Access All Manuscripts - to find terms semantically similar to a given concept in the context of medical literature.
This technique leverages the semantic relationships captured by the word embeddings to explore and understand the conceptual adjustments within the embedding space

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'
closest_terms = adjust_term_vector(model.wv,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model.wv, words)

## FastText - Common Crawl + Wikipedia - 300

fasttext is an extension of the Word2Vec model developed by Facebook AI Research. Unlike traditional Word2Vec, which considers words as atomic units, fasttext trates words as a bags of character n-grams. This approach allows fasttext to generate embeddings for out-of-vocabulary words by aggregating embeddings of their character n-grams, enabling it to handle rare or misspelled words effectively.

In addition to word embeddings, fasttext also supports sentence embeddings. It achieves this by averaging the word embeddings within a sentence to create a fixed-size vector representation of the entire sentence.

**fasttext word vectors for 157 languages overview**:

**Algorithm**: fasttext

**Training Data**: Wikipedia + Commn Crawl

**Vector Dimension**: 300 - dimension embedding

Let's download the model

In [ ]:
# !wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.zip
!wget http://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
!gzip -d cc.en.300.bin.gz

Let's load the model

In [ ]:
ft_model = fasttext.load_model('cc.en.300.bin')

Let's see how big the vocabulary:

In [ ]:
print(f'Size of the vocabulary: {len(ft_model.get_words())}')

### Inspecting embeddings and finding similar words

Return 300-dimensional vector representations of each word

In [ ]:
term = 'diabetes'
ft_model.get_word_vector(term)

In [ ]:
term = 'heart'
ft_model.get_word_vector(term)

In [ ]:
term = 'pregnancy'
ft_model.get_word_vector(term)

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space.


In [ ]:
term = 'delirium'
ft_model.get_nearest_neighbors(term)

In [ ]:
term = 'heart'
ft_model.get_nearest_neighbors(term)

The upcoming snippet of code demonstrates how to use vector arithmetic for word's meaning shifts based on analogies with a pre-trained word embedding model

In [ ]:
term_vec = 'king'
adjust_from_vec = 'woman'
adjust_to_vec = 'man'

closest_terms = ft_model.get_analogies(adjust_from_vec, adjust_to_vec, term_vec)
closest_terms

Let's combine the retrived words with the ones used to retrive them

In [ ]:
words = [word[1] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]

In [ ]:
words

Let's plot the embeddings in the 2D space using PCA

In [ ]:
words_vec = np.array([ft_model.get_word_vector(word) for word in words])
pca = PCA(n_components=2)
words_vec_2d = pca.fit_transform(words_vec)
plt.figure(figsize=(7, 7))
plt.scatter(words_vec_2d[:, 0], words_vec_2d[:, 1], edgecolors='k', c='g')
for word, (x, y) in zip(words, words_vec_2d):
    plt.text(x, y, word, fontsize=12)
plt.show()

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'

closest_terms = ft_model.get_analogies(adjust_from_vec,adjust_to_vec,term_vec)

In [ ]:
words = [word[1] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]

In [ ]:
words

In [ ]:
words_vec = np.array([ft_model.get_word_vector(word) for word in words])
pca = PCA(n_components=2)
words_vec_2d = pca.fit_transform(words_vec)
plt.figure(figsize=(7, 7))
plt.scatter(words_vec_2d[:, 0], words_vec_2d[:, 1], edgecolors='k', c='g')
for word, (x, y) in zip(words, words_vec_2d):
    plt.text(x, y, word, fontsize=12)
plt.show()

In the upcoming line of code, we compute the embeddings for the first 100 questions of the dataset using the FastText model

In [ ]:
word_vectors_questions = np.array([ft_model.get_word_vector(doc) for doc in question_df['question'][:100] ])

Let's proceed to compute the distance matrix of the vector embeddings for the first 100 questions. Subsequently, we'll visualize the similarity matrix using a heatmap. Heatmapes depicting distance metrics offer valuable insights into the similarities between elements and can aid in rudimentary clustering, identifying similar elements within the dataset

In [ ]:
distance_matrix = squareform(pdist(word_vectors_questions, metric='cosine'))
plt.figure(figsize=(100, 100))
sns.heatmap(1 - distance_matrix[:100, :100], annot=True, cmap='coolwarm', square=True)
plt.show()

The same can be done for the first 100 answers

In [ ]:
word_vectors_answers = np.array([ft_model.get_word_vector(doc) for doc in answer_df['answer'][:100] ])

In [ ]:
distance_matrix = squareform(pdist(word_vectors_answers, metric='cosine'))
plt.figure(figsize=(100, 100))
sns.heatmap(1 - distance_matrix[:100, :100], annot=True, cmap='coolwarm', square=True)
plt.show()

By inspecting the heatmap and manually examing the dataset, it becomes evident that there are many similar questions within the dataset. The heatmap reveals clusters of  closely related answers through regions of similar colors, indicating high similarity, Moreover, performing clustering on the dataset could further confirm and identify these groups of similar questions more systematically

 While this approach may not be as sophisticated as more advanced clustering algorithms, it can provide useful initial insights into the structure of the data and potential groupings of similar elements.

##Conclusions

The analysis clearly demonstrates that utilizing pre-trained word embedding models leads to enhanced performance. One compelling analysis underscoring this advantage is the word analogy test (within the medical domain). It vividly illutrastes that pre-trained models talilored to specific domains consistently outperform their conterparts